# Video Games Reviews: Sentiment Classification

## 1. Project Overview

### Problem Context
A marketplace with millions of product reviews cannot read every review by hand. Negative reviews are the most urgent ones: they point to broken products, shipping problems or misleading descriptions, and the sooner the product team sees them, the sooner they can act. A model that labels each new review as negative or positive makes it possible to route complaints to the right people without reading each review.

This project uses Amazon Reviews 2023, category Video Games, and keeps only reviews written up to 31 December 2019, as the assignment requires. The input is the review title and the review text. The label comes from the star rating: ratings 1 and 2 are negative, ratings 4 and 5 are positive, and rating 3 is dropped because it is ambiguous.

### Objective
Build a classifier that recognises negative and positive reviews equally well, not one that is only good on the larger class. Most reviews are positive, so a model that always answers positive would look accurate and still be useless for finding complaints.

### Evaluation Metric
The primary metric is **macro-F1**, the mean of the F1 score of the negative class and the F1 score of the positive class. Each class counts the same, no matter how many reviews it has. Precision, recall, the confusion matrix and the precision-recall curve of the negative class are reported as supporting evidence.

### Approach
1. Download the data and stream it to a compact file, so the large raw file is never loaded at once.
2. Audit the data, clean it with fixed rules, and explore it.
3. Decide the features, then compare a baseline against several models on the same validation scheme.
4. Choose the final model and decision threshold on validation data only.
5. Score the held-out test set once, at the end, and save the final model.

### Notebook Roadmap
| Section | Content |
|---|---|
| 1 | Project overview, setup and reproducibility |
| 2 | Dataset description, download and loading |
| 3 | Data quality assessment |
| 4 | Data preparation |
| 5 | Exploratory data analysis |
| 6 | Feature engineering |
| 7 | Baseline model |
| 8 | Model training |
| 9 | Model evaluation |
| 10 | Interpretation |
| 11 | Conclusion and recommendations |
| 12 | Saving and reloading the model |

### Setup & Reproducibility

In [7]:
import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import sklearn
from huggingface_hub import hf_hub_download

SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path('data')
MODEL_DIR = Path('models')
RAW_PATH = DATA_DIR / 'raw' / 'review_categories' / 'Video_Games.jsonl'
PARQUET_PATH = DATA_DIR / 'reviews_until_2019.parquet'
STATS_PATH = DATA_DIR / 'stream_stats.json'

CUTOFF = pd.Timestamp('2020-01-01', tz='UTC')
CUTOFF_MS = int(CUTOFF.timestamp() * 1000)

DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

print('python', platform.python_version(), '| pandas', pd.__version__,
      '| numpy', np.__version__, '| pyarrow', pa.__version__, '| scikit-learn', sklearn.__version__)

python 3.11.15 | pandas 3.1.1 | numpy 2.4.6 | pyarrow 26.0.1 | scikit-learn 1.9.1


## 2. Dataset Description and Source

The data is published by the McAuley Lab (University of California San Diego) as Amazon Reviews 2023.

Dataset Detail:
- Dataset Name              : Amazon Reviews 2023, category Video_Games
- Dataset Publisher         : McAuley Lab, UCSD
- Dataset Creators          : Yupeng Hou, Jiacheng Li, Zhankui He, An Yan, Xiusi Chen, Julian McAuley
- Project Page              : [https://amazon-reviews-2023.github.io](https://amazon-reviews-2023.github.io)
- Hugging Face              : [https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023)
- File Used                 : `raw/review_categories/Video_Games.jsonl`
- Reference                 : Hou et al. (2024), *Bridging Language and Items for Retrieval and Recommendation*
- Terms of Use              : see the dataset pages above

### 2.1 Dataset Description

Each line of the file is one review written as a JSON object. The file is large (several gigabytes), so it is read line by line and never loaded as a whole. Reviews written on or after 1 January 2020 are skipped while reading, so everything kept respects the assignment's cutoff of 31 December 2019. Rating 3 is **not** removed here. It stays in the stored data so that the audit in Section 3 can describe the full rating scale, and it is removed in Section 4.

Only the columns listed below are stored. The review images field is skipped because it is not used.

#### Data Dictionary

| Column | Description | Type | Role |
|---|---|---|---|
| `rating` | Star rating given by the user, from 1.0 to 5.0 | float | Source of the label |
| `title` | Title of the review | text | Model input |
| `text` | Body of the review | text | Model input |
| `timestamp` | Time of the review, in milliseconds since 1970-01-01 UTC | integer | Time split and cutoff |
| `helpful_vote` | Number of users who marked the review as helpful | integer | Analysis only, not a model input |
| `verified_purchase` | Whether the review comes from a verified purchase | boolean | Analysis only, not a model input |
| `parent_asin` | Identifier of the product (parent product) | text | Leakage checks and grouping |
| `asin` | Identifier of the product variant | text | Leakage checks |
| `user_id` | Identifier of the reviewer | text | Leakage checks and grouping |

Only `title` and `text` are used as model inputs, as the assignment states. The columns marked for analysis are used to understand the data and to check for leakage, and they never enter a model.

### 2.2 Download

The file is downloaded from Hugging Face once and cached in `data/`. Running the cell again reuses the existing file.

In [8]:
raw_file = hf_hub_download(
    repo_id='McAuley-Lab/Amazon-Reviews-2023',
    filename='raw/review_categories/Video_Games.jsonl',
    repo_type='dataset',
    local_dir=DATA_DIR,
)
assert Path(raw_file).resolve() == RAW_PATH.resolve()
print(f'{RAW_PATH} ({RAW_PATH.stat().st_size / 1e9:.2f} GB)')

data/raw/review_categories/Video_Games.jsonl (2.68 GB)


### 2.3 Streaming to Parquet

The raw file is read one line at a time. Reviews before the cutoff are collected in blocks of 250,000 and appended to a Parquet file, which keeps memory small and makes later loading fast. The step is skipped when the Parquet file already exists. Set `REBUILD = True` to run it again.

In [9]:
REBUILD = False
CHUNK_ROWS = 250_000

FIELDS = ['rating', 'title', 'text', 'timestamp', 'helpful_vote',
          'verified_purchase', 'parent_asin', 'asin', 'user_id']
SCHEMA = pa.schema([
    ('rating', pa.float32()),
    ('title', pa.string()),
    ('text', pa.string()),
    ('timestamp', pa.int64()),
    ('helpful_vote', pa.int32()),
    ('verified_purchase', pa.bool_()),
    ('parent_asin', pa.string()),
    ('asin', pa.string()),
    ('user_id', pa.string()),
])

if PARQUET_PATH.exists() and STATS_PATH.exists() and not REBUILD:
    print('Parquet file found, streaming skipped.')
else:
    tmp_path = PARQUET_PATH.with_suffix('.tmp')
    total = kept = 0
    buffer = []
    writer = pq.ParquetWriter(tmp_path, SCHEMA)

    def flush():
        writer.write_table(pa.Table.from_pylist(buffer, schema=SCHEMA))
        buffer.clear()

    with open(RAW_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            total += 1
            record = json.loads(line)
            if record['timestamp'] >= CUTOFF_MS:
                continue
            kept += 1
            buffer.append({k: record.get(k) for k in FIELDS})
            if len(buffer) == CHUNK_ROWS:
                flush()
                print(f'{total:,} lines read, {kept:,} kept', end='\r')
    if buffer:
        flush()
    writer.close()
    tmp_path.replace(PARQUET_PATH)
    STATS_PATH.write_text(json.dumps({'lines_read': total, 'kept_before_cutoff': kept}))
    print(f'{total:,} lines read, {kept:,} kept')

4,624,615 lines read, 3,044,032 kept


### 2.4 Loading the Data

The Parquet file is loaded into a data frame `df`. A readable `date` column is added for convenience, and the cutoff is verified.

In [10]:
stats = json.loads(STATS_PATH.read_text())
df = pd.read_parquet(PARQUET_PATH)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)

assert len(df) == stats['kept_before_cutoff']
assert df['date'].max() < CUTOFF

print(f"Lines in raw file          : {stats['lines_read']:,}")
print(f"Reviews before the cutoff  : {len(df):,}")
print(f"Share removed by cutoff    : {1 - len(df) / stats['lines_read']:.2%}")
print(f"Earliest review            : {df['date'].min()}")
print(f"Latest review              : {df['date'].max()}")
print(f"Memory used by df          : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Lines in raw file          : 4,624,615
Reviews before the cutoff  : 3,044,032
Share removed by cutoff    : 34.18%
Earliest review            : 1998-11-17 19:58:49+00:00
Latest review              : 2019-12-31 23:59:55.899000+00:00
Memory used by df          : 1.46 GB


In [11]:
df.head()

,rating,title,text,timestamp,helpful_vote,verified_purchase,parent_asin,asin,user_id,date
0,5.0000,... an order for my kids & they have really enjoyed playing this PC game,This was an order for my kids & they have really enjoyed playing this PC game.,1490877431000,0,True,B07MFMFW34,B01FEHJYUU,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,2017-03-30 12:37:11.000+00:00
1,5.0000,Great alt to pro controller,"These work great, They use batteries which is a bummer, but for the 40 less that i paid its wort...",1577637634017,0,True,B0BCHWZX95,B07GXJHRVK,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,2019-12-29 16:40:34.017+00:00
2,5.0000,solid product,I would recommend to anyone looking to add just a little bit of height and a lot of grip to thei...,1427591932000,0,True,B00HUWA45W,B00HUWA45W,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,2015-03-29 01:18:52.000+00:00
3,3.0000,love all the amazing colors but the black is really ...,love all the amazing colors but the black is really hard to see and I always have to have anothe...,1518124539574,0,True,B073SC6V1D,B016Y2BVKA,AHXSBZT52TCPZUBVCBRICTHWUCBA,2018-02-08 21:15:39.574+00:00
4,4.0000,good,"good, a tad bit too bulky, but good. they stay on. good texture too.",1523314072900,0,True,B001AW156U,B001AW156U,AG6BAEKWLCWH2TW3KKLVK773YF6A,2018-04-09 22:47:52.900+00:00


In [12]:
df.tail()

,rating,title,text,timestamp,helpful_vote,verified_purchase,parent_asin,asin,user_id,date
3044027,5.0000,Five Stars,The controller is a very good one that exceeded my expectations,1462757311000,0,True,B015NHBBOS,B015NHBBOS,AGN7AJKDEGY2TVZA6KF6ZIRQUUEA,2016-05-09 01:28:31.000+00:00
3044028,5.0000,Good seller,Works great,1468456314000,0,True,B015NHBBOS,B015NHBBOS,AFW3IME2QEZEQMNOJKP64Z2GT2KQ,2016-07-14 00:31:54.000+00:00
3044029,1.0000,DON'T WASTE YOUR MONEY!!!!,"From Hanoi Hanna blasting in you ear to ALL to errors in content, this is NOT a follow-up to Bat...",1080706836000,4,False,B0001AO01Y,B0001AO01Y,AFFYUOGDNI3JXGA22RSGHT2FJZ5A,2004-03-31 04:20:36.000+00:00
3044030,1.0000,Does not work with KVM switch,Nice keyboard but does not work with my KVM switch. Sending it back.,1531433542360,1,True,B074QM6F7X,B074QM6F7X,AEBGL7K4NBN2ALC6O353GJ3BIBYQ,2018-07-12 22:12:22.360+00:00
3044031,5.0000,Five Stars,"Worked like a charm! And cheap! We lost ours so, it's nice to be able to just get another!",1513198389319,0,True,B01GOYVNJM,B01GOYVNJM,AGDC4SOFDQ3UXPXMEYJ57SMM3F4A,2017-12-13 20:53:09.319+00:00


In [13]:
df.dtypes

rating                           float32
title                                str
text                                 str
timestamp                          int64
helpful_vote                       int32
verified_purchase                   bool
parent_asin                          str
asin                                 str
user_id                              str
date                 datetime64[us, UTC]
dtype: object